# Concevoir un autoencodeur convolutionnel (Keras/TensorFlow) — dataset `bottle`

Ce notebook construit un autoencodeur convolutionnel symétrique pour la détection d'anomalies sur le dataset `bottle` (128×128×3, pipeline `indusense.vision.dataset`), et détaille comment lire son `summary()`.

**Principe** : le modèle n'apprend qu'à reconstruire des images **saines**. Une fois entraîné, une image saine se reconstruit bien (faible erreur), une image défectueuse (jamais vue à l'entraînement) se reconstruit mal — l'erreur de reconstruction devient le score d'anomalie.

Sommaire :
1. Chargement du pipeline de données (réutilise `indusense.vision.dataset`)
2. Conception de l'architecture (encodeur / bottleneck / décodeur)
3. Construction du modèle et lecture du `summary()`
4. Vérification sur un vrai batch du pipeline
5. Compilation (loss, optimiseur) — prêt pour l'entraînement

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # réduit les logs TensorFlow

import sys
from pathlib import Path

# Localise la racine du projet en remontant depuis le cwd du kernel jusqu'à trouver pyproject.toml
# (plus robuste qu'un chemin relatif "src", qui suppose que le kernel a été démarré depuis la racine)
_project_root = Path.cwd()
while not (_project_root / "pyproject.toml").exists() and _project_root != _project_root.parent:
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root / "src"))

import tensorflow as tf
from tensorflow.keras import layers, Model

from indusense.vision.dataset import prepare_bottle_pipeline
from indusense.vision.augment import build_train_augmentations, make_augment_fn

## 1. Chargement du pipeline de données

On réutilise `prepare_bottle_pipeline` (déjà validé) pour obtenir directement des batches `(batch, 128, 128, 3)` normalisés `[0,1]`, avec augmentation sur l'entraînement uniquement.

In [ ]:
DATA_DIR = "data"   # <- à adapter à ton chemin local
IMG_SIZE = 128
BATCH_SIZE = 16

transform = build_train_augmentations()
augment_fn = make_augment_fn(transform)

pipeline = prepare_bottle_pipeline(
    data_dir=DATA_DIR, category="bottle",
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    augment_fn=augment_fn,
)
print("Train:", len(pipeline.train_paths), "images —", "Validation:", len(pipeline.val_paths), "images")

## 2. Conception de l'architecture

Un autoencodeur convolutionnel a deux moitiés symétriques :

- **Encodeur** : réduit progressivement la résolution spatiale (`strides=2`) tout en augmentant le nombre de filtres — échange l'information spatiale contre de l'information sémantique.
- **Bottleneck** : la représentation la plus compacte, au milieu du réseau — c'est la contrainte qui force le modèle à apprendre une structure du "normal" plutôt que de simplement copier l'image.
- **Décodeur** : miroir inverse (`Conv2DTranspose`), remonte en résolution jusqu'à retrouver la taille et le nombre de canaux de l'image d'origine.

**Choix de conception faits ici** :
- `strides=2` plutôt que `MaxPooling2D` : la convolution stridée apprend elle-même comment sous-échantillonner.
- Filtres qui doublent à chaque étage de l'encodeur (32→64→128→256) : compense la perte de résolution spatiale par plus de capacité de représentation.
- `padding="same"` partout : simplifie le calcul des tailles, garantit la symétrie encodeur/décodeur.
- Activation `sigmoid` en sortie : cohérente avec la normalisation `[0,1]` du pipeline (`indusense.vision.dataset.normalize`).

In [ ]:
def build_autoencoder(img_size: int = IMG_SIZE, base_filters: int = 32) -> Model:
    """Autoencodeur convolutionnel symétrique.

    Encodeur : 4 Conv2D stridées (downsampling ×2 à chaque étage).
    Bottleneck : représentation (img_size/16, img_size/16, base_filters*8).
    Décodeur : 4 Conv2DTranspose stridées (upsampling ×2 à chaque étage), miroir exact de l'encodeur.
    """
    inputs = layers.Input(shape=(img_size, img_size, 3), name="input_image")

    # --- Encodeur ---
    x = layers.Conv2D(base_filters,     3, strides=2, padding="same", activation="relu", name="enc_conv1")(inputs)
    x = layers.Conv2D(base_filters * 2, 3, strides=2, padding="same", activation="relu", name="enc_conv2")(x)
    x = layers.Conv2D(base_filters * 4, 3, strides=2, padding="same", activation="relu", name="enc_conv3")(x)
    x = layers.Conv2D(base_filters * 8, 3, strides=2, padding="same", activation="relu", name="enc_conv4")(x)

    latent = x  # bottleneck

    # --- Décodeur ---
    x = layers.Conv2DTranspose(base_filters * 4, 3, strides=2, padding="same", activation="relu", name="dec_conv1")(latent)
    x = layers.Conv2DTranspose(base_filters * 2, 3, strides=2, padding="same", activation="relu", name="dec_conv2")(x)
    x = layers.Conv2DTranspose(base_filters,     3, strides=2, padding="same", activation="relu", name="dec_conv3")(x)
    outputs = layers.Conv2DTranspose(3, 3, strides=2, padding="same", activation="sigmoid", name="dec_output")(x)

    return Model(inputs, outputs, name="conv_autoencoder")


autoencoder = build_autoencoder()

## 3. Lecture du `summary()`

Points à vérifier systématiquement dans le tableau affiché :

1. **Output Shape de la dernière couche == Output Shape de l'entrée** (`(None, 128, 128, 3)` des deux côtés) — condition nécessaire pour comparer la reconstruction à l'original pixel à pixel.
2. **Param # d'une `Conv2D`** : `(kernel_h × kernel_w × canaux_entrée + 1) × canaux_sortie` (le `+1` = biais par filtre).
3. **Symétrie du nombre de paramètres** entre encodeur et décodeur — un déséquilibre marqué signale souvent un bottleneck mal dimensionné.
4. **Total params vs volume de données** : à mettre en regard du nombre d'images d'entraînement pour anticiper un risque de sur-apprentissage.

In [ ]:
autoencoder.summary()

In [ ]:
# Vérification manuelle du calcul de Param # pour les deux premières couches (formule ci-dessus)
def conv2d_params(kernel, in_channels, out_channels):
    return (kernel * kernel * in_channels + 1) * out_channels

print("enc_conv1 attendu :", conv2d_params(3, 3, 32), "  (résultat summary() : 896)")
print("enc_conv2 attendu :", conv2d_params(3, 32, 64), " (résultat summary() : 18496)")

### Ratio de compression

Le bottleneck ne se juge pas qu'au nombre de paramètres — ce qui compte, c'est combien de **valeurs** il faut pour représenter une image une fois passée dans l'encodeur, comparé au nombre de valeurs de l'image d'origine :

```
ratio de compression = (nb de valeurs en entrée) / (nb de valeurs dans le latent)
                      = (H × W × C entrée) / (H' × W' × C' latent)
```

Un ratio élevé (bottleneck très compact) force le modèle à apprendre une représentation plus abstraite du "normal" — utile pour la détection d'anomalie, mais risque de sous-apprentissage (reconstruction dégradée même sur des images saines) si le ratio est trop agressif. Un ratio trop faible (bottleneck presque aussi grand que l'entrée) risque à l'inverse de laisser le modèle apprendre une quasi-identité, peu informative pour distinguer sain/défectueux.

In [ ]:
import numpy as np

input_shape = autoencoder.input.shape[1:]                 # (128, 128, 3)
latent_shape = autoencoder.get_layer("enc_conv4").output.shape[1:]  # (8, 8, 256), sortie du bottleneck

input_values = int(np.prod(input_shape))
latent_values = int(np.prod(latent_shape))
compression_ratio = input_values / latent_values

print(f"Entrée : {input_shape} -> {input_values} valeurs")
print(f"Latent : {latent_shape} -> {latent_values} valeurs")
print(f"Ratio de compression : {compression_ratio:.2f}x")

## 4. Vérification sur un vrai batch du pipeline

In [ ]:
for batch in pipeline.train_ds.take(1):
    reconstruction = autoencoder(batch)
    print("Batch d'entrée :", batch.shape)
    print("Reconstruction :", reconstruction.shape)

    mse = tf.keras.losses.MeanSquaredError()
    print("MSE (poids aléatoires, avant tout entraînement) :", float(mse(batch, reconstruction)))

In [ ]:
# Aperçu visuel : image d'entrée vs reconstruction (avant entraînement — reconstruction non informative à ce stade)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(batch[i].numpy())
    axes[0, i].set_title("Entrée")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstruction[i].numpy())
    axes[1, i].set_title("Reconstruction\n(avant entraînement)")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 5. Compilation (MSE + SSIM)

Pas de label : l'image d'entrée est aussi la cible (`fit(x, x)`).

- **Loss = MSE** (`mean_squared_error`) : pénalise l'erreur de reconstruction pixel à pixel, c'est elle qui pilote l'optimisation.
- **SSIM** (Structural Similarity Index) suivi en métrique additionnelle : contrairement au MSE, elle est sensible à la structure locale (luminance, contraste, texture) plutôt qu'à l'écart pixel à pixel brut — plus proche de la façon dont un défaut structurel (fissure, contamination) dégraderait perceptuellement la reconstruction. Keras n'a pas de métrique SSIM native, on l'encapsule via `tf.image.ssim`.
- **Optimiseur = Adam** : choix par défaut robuste pour ce type de modèle, pas de réglage fin du learning rate nécessaire pour démarrer.

In [ ]:
def ssim_metric(y_true, y_pred):
    """SSIM moyen sur le batch, encapsulé en métrique Keras (max_val=1.0 car images normalisées [0,1])."""
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

ssim_metric.__name__ = "ssim"  # nom utilisé dans history.history / les logs Keras

autoencoder.compile(optimizer="adam", loss="mse", metrics=[ssim_metric])

# Le pipeline retourne des images seules (x) ; l'entraînement d'un autoencodeur
# attend des paires (x, x) — target = entrée (reconstruction).
train_ds_xy = pipeline.train_ds.map(lambda x: (x, x))
val_ds_xy = pipeline.val_ds.map(lambda x: (x, x))

print("Modèle compilé — prêt pour l'entraînement (loss=mse, metric=ssim, optimizer=adam).")

## 6. Suivi MLflow

Même convention que le reste du projet (SQLite locale, `mlflow/mlflow.db`). Un run MLflow par entraînement, avec :
- **Params** loggés une fois (`img_size`, `batch_size`, `epochs`, `base_filters`, `optimizer`, `loss`).
- **Metrics** loggées à chaque epoch (`train_loss`, `val_loss`, `train_ssim`, `val_ssim`) — permet de comparer plusieurs runs dans l'UI MLflow (`mlflow ui --backend-store-uri sqlite:///mlflow/mlflow.db`).
- **Artifacts** : courbes d'apprentissage et exemples de reconstruction, sauvegardés en figures attachées au run.

In [ ]:
import mlflow

MLFLOW_TRACKING_URI = "sqlite:///mlflow/mlflow.db"  # <- adapter si besoin (cohérent avec le reste du projet)
EXPERIMENT_NAME = "bottle_autoencoder"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

## 7. Entraînement

`EPOCHS` volontairement bas par défaut pour une première validation rapide du pipeline — à augmenter une fois confirmé que loss et SSIM évoluent dans le bon sens.

In [ ]:
EPOCHS = 30

with mlflow.start_run(run_name="conv_autoencoder_bottle") as run:
    mlflow.log_params({
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "base_filters": 32,
        "optimizer": "adam",
        "loss": "mse",
    })

    history = autoencoder.fit(
        train_ds_xy,
        validation_data=val_ds_xy,
        epochs=EPOCHS,
        verbose=2,
    )

    # Log des métriques epoch par epoch (train/validation, loss + SSIM)
    for epoch in range(len(history.history["loss"])):
        mlflow.log_metrics({
            "train_loss": history.history["loss"][epoch],
            "val_loss": history.history["val_loss"][epoch],
            "train_ssim": history.history["ssim"][epoch],
            "val_ssim": history.history["val_ssim"][epoch],
        }, step=epoch)

    run_id = run.info.run_id

print(f"Run MLflow terminé : {run_id}")

## 8. Courbes d'apprentissage

Deux courbes pertinentes à suivre, chacune en train **et** validation (l'écart train/validation renseigne sur le sur-apprentissage) :
- **Loss (MSE)** : doit décroître sur les deux courbes ; un `val_loss` qui remonte alors que `train_loss` continue de baisser signale un sur-apprentissage.
- **SSIM** : doit croître vers 1.0 (reconstruction structurellement fidèle) ; à lire en complément de la loss, pas à sa place — les deux métriques peuvent diverger légèrement (le MSE pénalise l'écart brut, le SSIM la structure).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss (MSE)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["ssim"], label="train")
axes[1].plot(history.history["val_ssim"], label="validation")
axes[1].set_title("SSIM")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("SSIM")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

# Log de la figure comme artifact du run MLflow (run déjà terminé, on le rouvre par run_id)
with mlflow.start_run(run_id=run_id):
    mlflow.log_figure(fig, "curves.png")

plt.show()

## 9. Exemples de reconstruction après entraînement

Comparaison visuelle sur un batch de **validation** (jamais vu en augmentation, jamais utilisé pour la loss d'entraînement) : contrairement à l'aperçu de la section 4 (poids aléatoires), la reconstruction doit maintenant ressembler à l'original si l'entraînement a convergé.

In [ ]:
for batch in val_ds_xy.take(1):
    x_val, _ = batch
    reconstruction_trained = autoencoder(x_val)

n_show = min(4, x_val.shape[0])
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
for i in range(n_show):
    axes[0, i].imshow(x_val[i].numpy())
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstruction_trained[i].numpy())
    axes[1, i].set_title("Reconstruction\n(après entraînement)")
    axes[1, i].axis("off")
plt.tight_layout()

with mlflow.start_run(run_id=run_id):
    mlflow.log_figure(fig, "reconstructions.png")

plt.show()